# 🛡️ अल्टीमेट क्लाउड एंटीवायरस और प्राइवेसी सैंडबॉक्स

In [ ]:
import os, sys, shutil, glob, subprocess
from IPython.display import FileLink, display

# ------------------------------------------------------------
# 0. Sabhi zaroori tools install karo (full output dikhao)
# ------------------------------------------------------------
print('🔧 Tools install ho rahe hain...')
!apt-get update -y
!apt-get install -y clamav clamav-daemon unzip unrar p7zip-full wget file
!freshclam
print('✅ Sab tools tayar hain.\n')

print('🔧 mediafire-dl install kar rahe hain...')
!pip install -q git+https://github.com/Juvenal-Yescas/mediafire-dl.git
print('✅ mediafire-dl ready.\n')

# ------------------------------------------------------------
# 1. Link input
# ------------------------------------------------------------
print('🔒 Privacy active.')
mf_link = input('➡️ MediaFire ya file link paste karo aur Enter dabao: ')

# ------------------------------------------------------------
# 2. Download
# ------------------------------------------------------------
print('\n⏳ Downloading...')
# Purani archives hatao
for f in glob.glob('*.zip') + glob.glob('*.rar') + glob.glob('*.7z'):
    try:
        os.remove(f)
    except:
        pass
os.system(f'mediafire-dl "{mf_link}"')

# ------------------------------------------------------------
# 3. Downloaded file identify karo
# ------------------------------------------------------------
# Sabse nayi file dhundho (download ke baad)
all_files = [f for f in os.listdir('.') if os.path.isfile(f)]
if not all_files:
    print('❌ Koi file download nahi hui.')
    sys.exit(1)

# Newest file pick karo (download usually latest hoti hai)
newest = max(all_files, key=lambda f: os.path.getmtime(f))
print(f'📥 Downloaded file: {newest}')
print(f'📏 Size: {os.path.getsize(newest)} bytes')

# --- File type check ---
print('\n🔍 File type analysis:')
file_output = !file "{newest}"
print(file_output[0])

# --- Agar archive hai to andar ki file list dikhao ---
ext = newest.split('.')[-1].lower()
if ext in ['zip', 'rar', '7z']:
    print(f'\n📂 Archive contents of {newest}:')
    if ext == 'zip':
        !unzip -l "{newest}"
    elif ext == 'rar':
        !unrar l "{newest}"
    elif ext == '7z':
        !7z l "{newest}"
else:
    print(f'⚠️ Extension "{ext}" known archive format nahi hai. Phir bhi extract try karenge.')

# ------------------------------------------------------------
# 4. Extract
# ------------------------------------------------------------
print('\n⏳ Extracting...')
if os.path.exists('extracted_files'):
    shutil.rmtree('extracted_files')
os.makedirs('extracted_files', exist_ok=True)

if ext == 'zip':
    !unzip -o "{newest}" -d extracted_files/
elif ext == 'rar':
    !unrar x -o+ "{newest}" extracted_files/
elif ext == '7z':
    !7z x "{newest}" -oextracted_files/ -aoa
else:
    # Unknown format: phir bhi unzip try karo (kabhi kabhi extension galat hoti hai)
    !unzip -o "{newest}" -d extracted_files/ 2>/dev/null || echo 'Extraction failed – file valid archive nahi hai.'
    if not os.listdir('extracted_files'):
        print('❌ Extraction fail. File shayad corrupt hai ya supported format nahi hai.')
        sys.exit(1)
print('✅ Extraction complete.\n')

# ------------------------------------------------------------
# 5. Scan with ClamAV
# ------------------------------------------------------------
print('🛡️ Scanning...')
!clamscan --version
scan_result = !clamscan -r --remove extracted_files/

virus_found = False
for line in scan_result:
    if 'Infected files:' in line:
        print(line.strip())
        if 'Infected files: 0' not in line:
            virus_found = True

print('\n--- Scan Summary ---')
for line in scan_result[-10:]:
    print(line)

# ------------------------------------------------------------
# 6. Result & Download
# ------------------------------------------------------------
print('\n' + '='*50)
if virus_found:
    print('❌ Virus mila! File server se delete kar di gayi.')
    print('🛑 Download blocked!')
else:
    print('✅ File 100% safe hai. Clean zip bana rahe hain...')
    shutil.make_archive('safe_download', 'zip', 'extracted_files')
    print('📥 Safe download ready:')
    display(FileLink('safe_download.zip'))
print('='*50)